In [2]:
import pandas as pd
import csv
import joblib
import xgboost as xgb
from datetime import datetime
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
import seaborn as sns
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_fscore_support,
    matthews_corrcoef
)
import numpy as np
import tensorflow as tf
from tensorflow import keras
from imblearn.over_sampling import SMOTE
from collections import Counter
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from pyswarm import pso  # Librairie PSO



In [3]:
import pandas as pd
import os
import glob

# Chemin vers le dossier contenant les fichiers CSV
dossier = "../datasets"

# Utiliser glob pour récupérer tous les fichiers .csv
fichiers_csv = glob.glob(os.path.join(dossier, "*.csv"))

# Lire et concaténer tous les fichiers dans une liste
dataframes = [pd.read_csv(fichier) for fichier in fichiers_csv]

# Concaténer tous les DataFrames ensemble
df_concatene = pd.concat(dataframes, ignore_index=True)

# Afficher quelques infos
print(f"{len(fichiers_csv)} fichiers lus.")
print(f"Nombre total de lignes : {len(df_concatene)}")

# Enregistrer le résultat dans un nouveau fichier
df_concatene.to_csv("dataset_concatene.csv", index=False)

print("Tous les fichiers CSV ont été concaténés avec succès !")


139 fichiers lus.
Nombre total de lignes : 350126
Tous les fichiers CSV ont été concaténés avec succès !


In [4]:
import numpy as np
df_concatene.replace("-", np.nan, inplace=True)
print("Nombre total de valeurs nulles :", df_concatene.isnull().sum().sum())


Nombre total de valeurs nulles : 1212837


In [5]:
# Remplacer '-' par NaN
df_concatene.replace('-', np.nan, inplace=True)

# Remplacer les NaN par le mode (valeur la plus fréquente)
for column in df_concatene.columns:
    if df_concatene[column].isnull().any():
        mode_val = df_concatene[column].mode(dropna=True)
        if not mode_val.empty:
            df_concatene[column] = df_concatene[column].fillna(mode_val[0])  # ✅ Remplacement sans inplace


C:\Users\T U F\AppData\Local\Temp\ipykernel_5428\1019172921.py:7: UserWarning: Unable to sort modes: '<' not supported between instances of 'str' and 'float'
  mode_val = df_concatene[column].mode(dropna=True)


In [6]:
import numpy as np
df_concatene.replace("-", np.nan, inplace=True)
print("Nombre total de valeurs nulles :", df_concatene.isnull().sum().sum())


Nombre total de valeurs nulles : 0


In [7]:
df=df_concatene
print(df.dtypes)

Timestamp                object
Longitude               float64
Latitude                float64
Speed                     int64
Operatorname             object
CellID                    int64
NetworkMode              object
RSRP                      int64
RSRQ                     object
SNR                      object
CQI                      object
RSSI                     object
DL_bitrate                int64
UL_bitrate                int64
State                    object
NRxRSRP                  object
NRxRSRQ                  object
ServingCell_Lon          object
ServingCell_Lat          object
ServingCell_Distance     object
label                   float64
dtype: object


In [8]:
# Afficher les valeurs uniques pour chaque colonne de type object
for column in df.select_dtypes(include=['object']).columns:
    print(f"Valeurs uniques pour la colonne '{column}':")
    print(df[column].unique())
    print("\n")


Valeurs uniques pour la colonne 'Timestamp':
['2017.11.21_15.03.50' '2017.11.21_15.03.51' '2017.11.21_15.03.52' ...
 '2018.02.12_16.28.22' '2018.02.12_16.28.32' '2018.02.12_16.28.43']


Valeurs uniques pour la colonne 'Operatorname':
['A' '27205' '27202' '0' '27201' '27203' 'B']


Valeurs uniques pour la colonne 'NetworkMode':
['LTE' 'HSPA+' 'UMTS' 'EDGE' 'HSUPA' 'GPRS' 'HSDPA']


Valeurs uniques pour la colonne 'RSRQ':
[-13 -12 -14 -15 -17 -16 -18 -11 -19 -20 -10 -9 -8 -7 -6 '-12' '-11' '-13'
 '-15' '-14' '-17' '-18' '-2' '-20' '-8' '-9' '-7' '-10' '-19' '-21' '-23'
 '-24' '-22' '-16' '-6' '0' '-5' '2' '-3' '1' '-4' '5' '3' '6' -21 -2 -22
 -4 -5 -3 -24 -23 '4' '7']


Valeurs uniques pour la colonne 'SNR':
[4.0 2.0 13.0 -2.0 5.0 1.0 0.0 -4.0 -5.0 -3.0 3.0 -1.0 -10.0 -8.0 6.0 9.0
 8.0 10.0 12.0 15.0 7.0 11.0 -6.0 14.0 -9.0 -7.0 -13.0 -12.0 -16.0 17.0
 16.0 18.0 19.0 21.0 22.0 20.0 23.0 -11.0 '17' '8' '16' '19' '14' '7' '15'
 '12' '13' '11' '3' '5' '1' '2' '-3' '-30' '10' '4' '22' '6' '-

In [9]:
# Afficher les valeurs uniques de la colonne 'RSRQ'
unique_rsrq = df['RSRQ'].unique()
print(unique_rsrq)


[-13 -12 -14 -15 -17 -16 -18 -11 -19 -20 -10 -9 -8 -7 -6 '-12' '-11' '-13'
 '-15' '-14' '-17' '-18' '-2' '-20' '-8' '-9' '-7' '-10' '-19' '-21' '-23'
 '-24' '-22' '-16' '-6' '0' '-5' '2' '-3' '1' '-4' '5' '3' '6' -21 -2 -22
 -4 -5 -3 -24 -23 '4' '7']


In [10]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import pickle

# Chargement des données (à adapter si ton DataFrame est déjà chargé)
# df = pd.read_csv("ton_fichier.csv")

# 1. Extraction de l'année, mois, jour, heure, minute, seconde de 'Timestamp'
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%Y.%m.%d_%H.%M.%S')
df['year'] = df['Timestamp'].dt.year
df['month'] = df['Timestamp'].dt.month
df['day'] = df['Timestamp'].dt.day
df['hour'] = df['Timestamp'].dt.hour
df['minute'] = df['Timestamp'].dt.minute
df['second'] = df['Timestamp'].dt.second

# 2. Label Encoding pour 'Operatorname' et 'NetworkMode' + sauvegarde
le_operator = LabelEncoder()
df['Operatorname'] = le_operator.fit_transform(df['Operatorname'])
with open('le_operator.pkl', 'wb') as f:
    pickle.dump(le_operator, f)

le_network = LabelEncoder()
df['NetworkMode'] = le_network.fit_transform(df['NetworkMode'])
with open('le_network.pkl', 'wb') as f:
    pickle.dump(le_network, f)

# 3. Casting vers int pour 'RSRQ' et 'RSSI'
# Supprimer les espaces et convertir les valeurs en float
df['RSRQ'] = pd.to_numeric(df['RSRQ'].str.strip(), errors='coerce')

# Afficher les valeurs uniques après la conversion
unique_rsrq = df['RSRQ'].unique()
print(unique_rsrq)# Convertir directement en Int64

# Supprimer les espaces et convertir les valeurs en float
df['RSSI'] = pd.to_numeric(df['RSSI'].str.strip(), errors='coerce')

# Afficher les valeurs uniques après la conversion
unique_rsrq = df['RSSI'].unique()
print(unique_rsrq)

# 4. Casting vers float pour 'SNR'
df['SNR'] = pd.to_numeric(df['SNR'], errors='coerce')  # Conversion vers float, gestion des erreurs

# 5. Convertir les valeurs de 'CQI' en entiers (certaines sont string)
df['CQI'] = df['CQI'].astype(int)

# 6. Encodage de la colonne 'State' + sauvegarde
le_state = LabelEncoder()
df['State'] = le_state.fit_transform(df['State'])
with open('le_state.pkl', 'wb') as f:
    pickle.dump(le_state, f)

# 7. Conversion de 'NRxRSRP' et 'NRxRSRQ' en float
df['NRxRSRP'] = df['NRxRSRP'].astype(float)
df['NRxRSRQ'] = df['NRxRSRQ'].astype(float)

# 8. Conversion de 'ServingCell_Lon' et 'ServingCell_Lat' en float
df['ServingCell_Lon'] = df['ServingCell_Lon'].astype(float)
df['ServingCell_Lat'] = df['ServingCell_Lat'].astype(float)

# Conversion de 'ServingCell_Distance' en numérique, avec gestion des erreurs
df['ServingCell_Distance'] = pd.to_numeric(df['ServingCell_Distance'], errors='coerce')

# Vérifier les types après conversion
print(df.dtypes)


[ nan -12. -11. -13. -15. -14. -17. -18.  -2. -20.  -8.  -9.  -7. -10.
 -19. -21. -23. -24. -22. -16.  -6.   0.  -5.   2.  -3.   1.  -4.   5.
   3.   6.   4.   7.]
[-80. -78. -82. -85. -83. -81. -86. -84. -88. -87. -89. -90. -92. -94.
 -93. -91. -75. -77. -79. -74. -76. -69. -73. -72. -70. -71. -68. -65.
 -67. -66. -63. -64. -62. -57. -60. -59. -58. -61. -55. -56. -51. -54.
 -53. -52. -49. -46. -48. -45. -47. -50. -44. -43. -42. -41. -40.  nan
 -37. -39. -36. -38.]
Timestamp               datetime64[ns]
Longitude                      float64
Latitude                       float64
Speed                            int64
Operatorname                     int64
CellID                           int64
NetworkMode                      int64
RSRP                             int64
RSRQ                           float64
SNR                            float64
CQI                              int64
RSSI                           float64
DL_bitrate                       int64
UL_bitrate             

In [11]:
# Obtenir la valeur la plus fréquente (mode) de la colonne 'RSRQ'
mode_value = df['RSRQ'].mode()[0]

# Remplacer les NaN par la valeur la plus fréquente
df['RSRQ'] = df['RSRQ'].fillna(mode_value)

# Vérifier les résultats
print(df['RSRQ'].isna().sum())  # Devrait afficher 0, indiquant qu'il n'y a plus de NaN


0


In [12]:
print(df.dtypes)

Timestamp               datetime64[ns]
Longitude                      float64
Latitude                       float64
Speed                            int64
Operatorname                     int64
CellID                           int64
NetworkMode                      int64
RSRP                             int64
RSRQ                           float64
SNR                            float64
CQI                              int64
RSSI                           float64
DL_bitrate                       int64
UL_bitrate                       int64
State                            int64
NRxRSRP                        float64
NRxRSRQ                        float64
ServingCell_Lon                float64
ServingCell_Lat                float64
ServingCell_Distance           float64
label                          float64
year                             int32
month                            int32
day                              int32
hour                             int32
minute                   

In [13]:
# Afficher les valeurs uniques pour chaque colonne de type object
for column in df.select_dtypes(include=['float64']).columns:
    print(f"Valeurs uniques pour la colonne '{column}':")
    print(df[column].unique())
    print("\n")


Valeurs uniques pour la colonne 'Longitude':
[-8.499701 -8.499733 -8.499887 ... -8.499397 -8.484568 -8.500138]


Valeurs uniques pour la colonne 'Latitude':
[51.893336 51.893346 51.893384 ... 51.893436 51.896165 51.893262]


Valeurs uniques pour la colonne 'RSRQ':
[ -2. -12. -11. -13. -15. -14. -17. -18. -20.  -8.  -9.  -7. -10. -19.
 -21. -23. -24. -22. -16.  -6.   0.  -5.   2.  -3.   1.  -4.   5.   3.
   6.   4.   7.]


Valeurs uniques pour la colonne 'SNR':
[  4.   2.  13.  -2.   5.   1.   0.  -4.  -5.  -3.   3.  -1. -10.  -8.
   6.   9.   8.  10.  12.  15.   7.  11.  -6.  14.  -9.  -7. -13. -12.
 -16.  17.  16.  18.  19.  21.  22.  20.  23. -11. -30. -14. -20. -19.
 -17. -18. -15.  25.  28.  26.  27.  24.  31.  29.  32.  30.  33.]


Valeurs uniques pour la colonne 'RSSI':
[-80. -78. -82. -85. -83. -81. -86. -84. -88. -87. -89. -90. -92. -94.
 -93. -91. -75. -77. -79. -74. -76. -69. -73. -72. -70. -71. -68. -65.
 -67. -66. -63. -64. -62. -57. -60. -59. -58. -61. -55. -56. -51. -54.


In [14]:

seuil_debit = 2000  # ↓ plus bas qu'avant (avant c'était 1000)
seuil_rsrp = -97    # ↓ un peu plus tolérant
seuil_rsrq = -14   # = médiane, donc bon pour couper en deux

# Créer une fonction qui attribue le label en fonction des conditions
def attribuer_label(row):
    if (row['DL_bitrate'] > seuil_debit or row['UL_bitrate'] > seuil_debit) or (row['RSRP'] < seuil_rsrp and row['RSRQ'] < seuil_rsrq):
        return 1
    else:
        return 0


# Appliquer la fonction sur chaque ligne du dataset
df['label'] = df.apply(attribuer_label, axis=1)

# Enregistrer le nouveau dataset avec la colonne 'label' dans un fichier CSV
df.to_csv('dataSetWithLabel.csv', index=False)

print("Le nouveau dataset a été créé avec succès.")



Le nouveau dataset a été créé avec succès.


In [15]:

valeurs_uniques = df['label'].unique()
print(valeurs_uniques)

[0 1]


In [16]:
valeurs_par_classe = df['label'].value_counts()
print(valeurs_par_classe)


label
1    254967
0     95159
Name: count, dtype: int64


Data Augmentation

In [17]:
df = df.drop(columns=df.select_dtypes(include=['datetime']).columns)

In [18]:
df.head()

,Longitude,Latitude,Speed,Operatorname,CellID,NetworkMode,RSRP,RSRQ,SNR,CQI,...,ServingCell_Lon,ServingCell_Lat,ServingCell_Distance,label,year,month,day,hour,minute,second
0,-8.499701,51.893336,0,5,2,5,-95,-2.0,4.0,10,...,-8.491719,51.893905,551.37,0,2017,11,21,15,3,50
1,-8.499701,51.893336,0,5,2,5,-95,-2.0,2.0,8,...,-8.491719,51.893905,551.37,0,2017,11,21,15,3,51
2,-8.499733,51.893346,0,5,2,5,-95,-2.0,13.0,9,...,-8.491719,51.893905,553.43,0,2017,11,21,15,3,52
3,-8.499733,51.893346,0,5,2,5,-95,-2.0,13.0,9,...,-8.491719,51.893905,553.43,0,2017,11,21,15,3,52
4,-8.499887,51.893384,1,5,2,5,-95,-2.0,13.0,9,...,-8.491719,51.893905,563.48,0,2017,11,21,15,3,53


In [19]:
print(df.dtypes)

Longitude               float64
Latitude                float64
Speed                     int64
Operatorname              int64
CellID                    int64
NetworkMode               int64
RSRP                      int64
RSRQ                    float64
SNR                     float64
CQI                       int64
RSSI                    float64
DL_bitrate                int64
UL_bitrate                int64
State                     int64
NRxRSRP                 float64
NRxRSRQ                 float64
ServingCell_Lon         float64
ServingCell_Lat         float64
ServingCell_Distance    float64
label                     int64
year                      int32
month                     int32
day                       int32
hour                      int32
minute                    int32
second                    int32
dtype: object


In [20]:
# Remplacer plusieurs valeurs manquantes potentielles par np.nan
df.replace(["-", "", "NULL", "NaN", "null", "N/A"], np.nan, inplace=True)

# Vérifier de nouveau le nombre de valeurs nulles
print("Nombre total de valeurs nulles :", df.isnull().sum().sum())


Nombre total de valeurs nulles : 4118


In [21]:
# Remplacer les NaN dans chaque colonne par la valeur la plus fréquente (mode)
for column in df.columns:
    mode_value = df[column].mode()[0]  # Obtenir la valeur la plus fréquente
    df[column] = df[column].fillna(mode_value)  # Réassigner la colonne après remplacement

# Vérifier de nouveau le nombre de valeurs nulles
print("Nombre total de valeurs nulles après remplacement :", df.isnull().sum().sum())


Nombre total de valeurs nulles après remplacement : 0


In [22]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder
from collections import Counter
import pandas as pd

# Vérifier les classes avant SMOTE
print("Avant SMOTE :", Counter(df["label"]))

# Caractéristiques
X = df.drop(columns=["label"])

# SMOTE classique pour binaire : on veut suréchantillonner la classe minoritaire (0)
smote = SMOTE(sampling_strategy='auto', k_neighbors=2, random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, df["label"])

# Vérifier les classes après SMOTE
print("Après SMOTE :", Counter(y_resampled))


# Reconstruction du DataFrame
balanced_df = pd.DataFrame(X_resampled, columns=X.columns)
balanced_df["label"] = y_resampled # ou directement y_resampled si tu veux garder 0/1


Avant SMOTE : Counter({1: 254967, 0: 95159})
Après SMOTE : Counter({0: 254967, 1: 254967})


In [23]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import joblib
import xgboost as xgb

# Séparation des données
X = balanced_df.drop(columns=["label"])
y = balanced_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Initialiser et ajuster MinMaxScaler
minmax_scaler = MinMaxScaler()
X_train_scaled = minmax_scaler.fit_transform(X_train)
X_test_scaled = minmax_scaler.transform(X_test)

In [26]:
# Convertir les données mises à l'échelle en DataFrame
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)
# Enregistrer les fichiers sous format CSV
X_train_scaled_df.to_csv('X_train.csv', index=False)
X_test_scaled_df.to_csv('X_test.csv', index=False)

# Enregistrer aussi les labels sous format CSV
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

In [ ]:


# ✅ Enregistrer le scaler
joblib.dump(minmax_scaler, "minmax_scaler1.pkl")

# Initialiser le modèle XGBoost pour classification binaire
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1
)

# Entraîner le modèle
xgb_model.fit(X_train_scaled, y_train)

print("\n✅ Modèle XGBoost entraîné avec MinMaxScaler et scaler enregistré dans 'minmax_scaler.pkl'")


c:\Users\T U F\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:51:18] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



✅ Modèle XGBoost entraîné avec MinMaxScaler et scaler enregistré dans 'minmax_scaler.pkl'


In [202]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    confusion_matrix, roc_auc_score, matthews_corrcoef
)
import numpy as np

# 🎯 1. Prédiction
y_pred = xgb_model.predict(X_test_scaled)
y_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]  # Probabilité de la classe positive (1)

# ✅ 2. Métriques globales
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

# 🔍 3. Confusion Matrix & FPR (False Positive Rate)
cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()
FPR = FP / (FP + TN) if (FP + TN) > 0 else 0

# 📊 4. ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_proba)

# 🔢 5. Affichage des métriques
print(f"✅ Accuracy : {accuracy:.4f}")
print(f"🎯 Precision : {precision:.4f}")
print(f"🔁 Recall : {recall:.4f}")
print(f"⚡ F1-Score : {f1:.4f}")
print(f"🚨 FAR / FPR : {FPR:.4f}")
print(f"📈 ROC-AUC Score : {roc_auc:.4f}")
print(f"🧩 Matthews Correlation Coefficient (MCC) : {mcc:.4f}")


✅ Accuracy : 0.9992
🎯 Precision : 0.9991
🔁 Recall : 0.9994
⚡ F1-Score : 0.9992
🚨 FAR / FPR : 0.0009
📈 ROC-AUC Score : 1.0000
🧩 Matthews Correlation Coefficient (MCC) : 0.9984


In [203]:
joblib.dump(xgb_model, "../savemodels/xgboost_modelv1.pkl")
print("\n💾 Modèle sauvegardé sous 'xgboost_modelv1.pkl'.")



💾 Modèle sauvegardé sous 'xgboost_modelv1.pkl'.


In [204]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, matthews_corrcoef

# ✅ Reshape pour LSTM (samples, timesteps, features)
X_train_reshaped = np.expand_dims(X_train_scaled, axis=1)
X_test_reshaped = np.expand_dims(X_test_scaled, axis=1)

# 1️⃣ Modèle LSTM pour classification binaire
model = Sequential([
    LSTM(100, activation='tanh', return_sequences=True, input_shape=(1, X_train_scaled.shape[1])),
    Dropout(0.2),
    LSTM(50, activation='tanh', return_sequences=False),
    Dense(25, activation='relu'),
    Dense(1, activation='sigmoid')  # Sigmoid pour binaire
])

# 2️⃣ Compilation
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 3️⃣ Entraînement
model.fit(X_train_reshaped, y_train, epochs=30, batch_size=32, validation_data=(X_test_reshaped, y_test))

# 4️⃣ Prédictions
y_pred_probs = model.predict(X_test_reshaped).flatten()  # Probabilités de la classe positive
y_pred = (y_pred_probs >= 0.5).astype(int)  # Seuil à 0.5

# 5️⃣ Évaluation
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

# 🔍 FPR / FAR
cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()
FPR = FP / (FP + TN) if (FP + TN) > 0 else 0

# 📈 ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_pred_probs)

# 🔢 Affichage
print(f"✅ Accuracy : {accuracy:.4f}")
print(f"🎯 Precision : {precision:.4f}")
print(f"🔁 Recall : {recall:.4f}")
print(f"⚡ F1-Score : {f1:.4f}")
print(f"🚨 FAR / FPR : {FPR:.4f}")
print(f"📈 ROC-AUC Score : {roc_auc:.4f}")
print(f"🧩 Matthews Correlation Coefficient (MCC) : {mcc:.4f}")


c:\Users\T U F\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/30
6118/6118 ━━━━━━━━━━━━━━━━━━━━ 29s 4ms/step - accuracy: 0.9042 - loss: 0.2117 - val_accuracy: 0.9784 - val_loss: 0.0646
Epoch 2/30
6118/6118 ━━━━━━━━━━━━━━━━━━━━ 25s 4ms/step - accuracy: 0.9732 - loss: 0.0642 - val_accuracy: 0.9818 - val_loss: 0.0410
Epoch 3/30
6118/6118 ━━━━━━━━━━━━━━━━━━━━ 25s 4ms/step - accuracy: 0.9798 - loss: 0.0499 - val_accuracy: 0.9872 - val_loss: 0.0325
Epoch 4/30
6118/6118 ━━━━━━━━━━━━━━━━━━━━ 25s 4ms/step - accuracy: 0.9823 - loss: 0.0418 - val_accuracy: 0.9856 - val_loss: 0.0326
Epoch 5/30
6118/6118 ━━━━━━━━━━━━━━━━━━━━ 25s 4ms/step - accuracy: 0.9835 - loss: 0.0399 - val_accuracy: 0.9831 - val_loss: 0.0367
Epoch 6/30
6118/6118 ━━━━━━━━━━━━━━━━━━━━ 25s 4ms/step - accuracy: 0.9839 - loss: 0.0385 - val_accuracy: 0.9903 - val_loss: 0.0241
Epoch 7/30
6118/6118 ━━━━━━━━━━━━━━━━━━━━ 25s 4ms/step - accuracy: 0.9860 - loss: 0.0340 - val_accuracy: 0.9881 - val_loss: 0.0274
Epoch 8/30
6118/6118 ━━━━━━━━━━━━━━━━━━━━ 25s 4ms/step - accuracy: 0.9858 - loss: 0

In [205]:
joblib.dump(model, "../savemodels/lstmv1.pkl")
print("\n💾 Modèle sauvegardé sous 'lstmv1.pkl'.")



💾 Modèle sauvegardé sous 'lstmv1.pkl'.


In [28]:
from sklearn.linear_model import LogisticRegression

# Initialiser le modèle de Régression Logistique pour classification binaire
logreg_model = LogisticRegression(
    solver='liblinear',  # Choisir le solver 'liblinear' pour les petits jeux de données
    random_state=42,     # Fixer la graine pour la reproductibilité
    max_iter=1000        # Augmenter le nombre d'itérations si nécessaire
)

# Entraîner le modèle
logreg_model.fit(X_train_scaled, y_train)

# Prédiction
y_pred_logreg = logreg_model.predict(X_test_scaled)
y_proba_logreg = logreg_model.predict_proba(X_test_scaled)[:, 1]  # Probabilité de la classe positive (1)

# Métriques globales
accuracy_logreg = accuracy_score(y_test, y_pred_logreg)
precision_logreg = precision_score(y_test, y_pred_logreg)
recall_logreg = recall_score(y_test, y_pred_logreg)
f1_logreg = f1_score(y_test, y_pred_logreg)
mcc_logreg = matthews_corrcoef(y_test, y_pred_logreg)

# Confusion Matrix & FPR (False Positive Rate)
cm_logreg = confusion_matrix(y_test, y_pred_logreg)
TN_logreg, FP_logreg, FN_logreg, TP_logreg = cm_logreg.ravel()
FPR_logreg = FP_logreg / (FP_logreg + TN_logreg) if (FP_logreg + TN_logreg) > 0 else 0

# ROC-AUC Score
roc_auc_logreg = roc_auc_score(y_test, y_proba_logreg)

# Affichage des métriques
print(f"✅ Accuracy : {accuracy_logreg:.4f}")
print(f"🎯 Precision : {precision_logreg:.4f}")
print(f"🔁 Recall : {recall_logreg:.4f}")
print(f"⚡ F1-Score : {f1_logreg:.4f}")
print(f"🚨 FAR / FPR : {FPR_logreg:.4f}")
print(f"📈 ROC-AUC Score : {roc_auc_logreg:.4f}")
print(f"🧩 Matthews Correlation Coefficient (MCC) : {mcc_logreg:.4f}")


✅ Accuracy : 0.9482
🎯 Precision : 0.9767
🔁 Recall : 0.9184
⚡ F1-Score : 0.9466
🚨 FAR / FPR : 0.0219
📈 ROC-AUC Score : 0.9915
🧩 Matthews Correlation Coefficient (MCC) : 0.8981


In [29]:
from sklearn.neural_network import MLPClassifier

# Initialiser le modèle MLP pour classification binaire
mlp_model = MLPClassifier(
    hidden_layer_sizes=(100,),   # Taille du réseau de neurones (ici une couche cachée de 100 neurones)
    activation='relu',           # Fonction d'activation ReLU
    solver='adam',               # Optimiseur Adam, efficace pour la plupart des cas
    max_iter=1000,               # Nombre maximal d'itérations pour la convergence
    random_state=42,             # Fixer la graine pour la reproductibilité
    n_iter_no_change=10,         # Nombre d'itérations sans amélioration avant d'arrêter
)

# Entraîner le modèle
mlp_model.fit(X_train_scaled, y_train)

# Prédiction
y_pred_mlp = mlp_model.predict(X_test_scaled)
y_proba_mlp = mlp_model.predict_proba(X_test_scaled)[:, 1]  # Probabilité de la classe positive (1)

# Métriques globales
accuracy_mlp = accuracy_score(y_test, y_pred_mlp)
precision_mlp = precision_score(y_test, y_pred_mlp)
recall_mlp = recall_score(y_test, y_pred_mlp)
f1_mlp = f1_score(y_test, y_pred_mlp)
mcc_mlp = matthews_corrcoef(y_test, y_pred_mlp)

# Confusion Matrix & FPR (False Positive Rate)
cm_mlp = confusion_matrix(y_test, y_pred_mlp)
TN_mlp, FP_mlp, FN_mlp, TP_mlp = cm_mlp.ravel()
FPR_mlp = FP_mlp / (FP_mlp + TN_mlp) if (FP_mlp + TN_mlp) > 0 else 0

# ROC-AUC Score
roc_auc_mlp = roc_auc_score(y_test, y_proba_mlp)

# Affichage des métriques
print(f"✅ Accuracy : {accuracy_mlp:.4f}")
print(f"🎯 Precision : {precision_mlp:.4f}")
print(f"🔁 Recall : {recall_mlp:.4f}")
print(f"⚡ F1-Score : {f1_mlp:.4f}")
print(f"🚨 FAR / FPR : {FPR_mlp:.4f}")
print(f"📈 ROC-AUC Score : {roc_auc_mlp:.4f}")
print(f"🧩 Matthews Correlation Coefficient (MCC) : {mcc_mlp:.4f}")


✅ Accuracy : 0.9948
🎯 Precision : 0.9996
🔁 Recall : 0.9901
⚡ F1-Score : 0.9948
🚨 FAR / FPR : 0.0004
📈 ROC-AUC Score : 1.0000
🧩 Matthews Correlation Coefficient (MCC) : 0.9897


In [30]:
joblib.dump(mlp_model, "../savemodels/MLPv1.pkl")
print("\n💾 Modèle sauvegardé sous 'mlpv1.pkl'.")



💾 Modèle sauvegardé sous 'mlpv1.pkl'.
